# 04 · A SQL assistant: decide around the queries

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daseinlabs/open-jev/blob/main/cookbooks/04-sql-assistant/cookbook.ipynb)


Your chat assistant answers questions by writing SQL (a handful of queries per conversation),
and you log whether each answer was right. Fine-tune a decision model on those logs so it takes
the decisions **around** the SQL:

| decision | candidates |
|---|---|
| `action` | run another query (`run_sql`) or answer now (`respond`) |
| `tables.sql` | which tables the next query needs: every set of tables joinable through foreign keys (up to 3), or `schema` (look at the catalog) |

Your LLM still writes the SQL, given only the chosen tables. The model routes, with a
calibrated confidence, and defers to your LLM when unsure.


**Two methods**, chosen with `METHOD` in the setup cell; everything else is the same:

- `'tod'` (default) fine-tunes [TOD](https://huggingface.co/parsecai/tod) (Gemma 4 12B + LoRA). Needs an
  NVIDIA GPU with 40 GB+ (A100, H100, L40S; on Colab an A100 runtime) or a 64 GB Apple silicon Mac.
  Downloads `google/gemma-4-12B-it` (24 GB, not gated) once.
- `'head'` trains small heads on a frozen Gemma 3 4B. Needs ~9 GB (any recent NVIDIA GPU, or any Apple
  silicon Mac). Downloads [`parsecai/gemma-3-4b-it`](https://huggingface.co/parsecai/gemma-3-4b-it)
  (8 GB, gated: accept the Gemma terms there first).

In [ ]:
METHOD = 'tod'  # 'tod': fine-tune TOD (40 GB+ GPU) | 'head': small heads on Gemma 3 4B (~9 GB)

# Setup: install the SDK if it's missing, log in to Hugging Face once (for the gated Gemma 3 backbone).
import importlib.util
if importlib.util.find_spec('openjev') is None:
    %pip install -q openjev-sdk

from huggingface_hub import get_token, login
if METHOD == 'head' and get_token() is None:
    login()

import json, shutil
from pathlib import Path
from openjev import compute
from openjev.tod.picker import finetune, release
from openjev.tod.rl import load_policy
from openjev.tod.train import train_dataset

def train(dataset, out):
    # fine-tune TOD, or train heads; then calibrate and report test accuracy per kind
    return finetune(dataset, out) if METHOD == 'tod' else train_dataset(dataset, None, out)

def fresh(path):
    """Remove a previous run's output, so every cell can be re-run."""
    shutil.rmtree(path, ignore_errors=True)
    return Path(path)

print('method:', METHOD, '| device:', compute.device() if METHOD == 'tod' or compute.backend() == 'torch' else 'metal (MLX)')

## 1. Traces

The simulated assistant (`openjev.tod.envs.sqldesk`) answers questions about a shop database in
SQLite: it lists tables, peeks at rows, runs the queries the question needs, sometimes
double-checks, and answers. It is right about 90% of the time; it sometimes forgets a join, or
answers one query too early.

**For your data, point `TRACES` at your logs** (OpenAI tool-calling format with an `outcome`
per conversation) and skip the generation.

In [ ]:
from openjev.tod.envs.sqldesk import SCHEMA, record_trace

RUN = Path('runs/sql')
RUN.mkdir(parents=True, exist_ok=True)
TRACES = RUN/'traces.jsonl'
if not TRACES.exists():
    with TRACES.open('w') as stream:
        stream.writelines(json.dumps(record_trace(f'chat-{i:05d}', seed=i * 2)) + '\n' for i in range(1500))

traces = [json.loads(line) for line in TRACES.open() if line.strip()]
queries = sum(1 for t in traces for m in t['messages'] if m.get('tool_calls')) / len(traces)
print(f"{len(traces)} traces, {queries:.1f} queries per conversation, success {sum(t['outcome'] for t in traces) / len(traces):.1%}\n")
for m in traces[3]['messages']:
    print(f"{m['role']:9}", m.get('content') or json.loads(m['tool_calls'][0]['function']['arguments'])['sql'])

## 2. Schema + traces → decisions

The schema is `{table: {columns: [...], references: [tables]}}`. With your database:
`schema_from_sqlite(sqlite3.connect('my.db'))`, or the same dict built from `information_schema`.

Labels come straight from the logs: `action` is whether the next message was a query or the
answer; `tables.sql` is the set of tables the logged SQL reads (from its `FROM` / `JOIN`
clauses). Read the report:

- `recall_miss_rate` for `tables.sql`: queries whose tables aren't a candidate (a bigger join,
  or tables with no foreign key between them). Raise `max_tables` or add the `references`.
- `duplicates_with_different_label`: logs repeat (the same question, the same first step), and
  the assistant doesn't always do the same thing. That part of the decision is a coin flip.

In [ ]:
from openjev.tod import build_dataset
from openjev.tod.chat_traces import episodes_from_traces, spec_from_tools, table_sets

spec = spec_from_tools('sql-assistant', traces[0]['tools'], sql_params={'sql': dict(schema=SCHEMA, max_tables=3)})
print(len(table_sets(SCHEMA)) + 1, 'table-set candidates, e.g.', [c.id for c in spec.enumerate('tables.sql', {})][:6])
episodes, coverage = episodes_from_traces(traces, spec)
dataset = build_dataset(spec, episodes, min_outcome=1.0)

DATASET = fresh(RUN/'dataset')
dataset.write(DATASET)
(RUN/'spec.json').unlink(missing_ok=True)
spec.save(RUN/'spec.json')   # serving rebuilds the identical spec from this file
print(json.dumps({k: dataset.report[k] for k in ('splits', 'recall_miss_rate', 'duplicates_dropped',
                                                 'duplicates_with_different_label')}, indent=1))

## 3. Fine-tune

In [ ]:
MODEL = fresh(RUN/'model')
result = train(DATASET, MODEL)
for kind, r in result['test'].items():
    print(f"{kind:11} accuracy {r['accuracy']:.1%}   mean confidence {r['mean_confidence']:.2f}   n={r['n']}")

## 4. Let it drive the assistant

On 200 fresh questions, the model decides when to query and which tables; a simulated LLM
writes the SQL for those tables (right when they cover what the question needs, a plausible
wrong query when a needed table is missing). Deferred decisions go to the parent: an oracle
here, your LLM in production.

In [ ]:
import random
from openjev.tod.chat_traces import ChatTraceSpec
from openjev.tod.envs.sqldesk import SqlDesk, teacher

SEEDS = range(10_000_000, 10_000_200)

def assistant_success(seed):
    desk = SqlDesk().reset(seed)
    teacher(desk, random.Random(seed + 1))
    return desk.success

spec = ChatTraceSpec.load(RUN/'spec.json')
policy = load_policy(MODEL, spec)
print(f'assistant that wrote the traces: success {sum(map(assistant_success, SEEDS)) / len(SEEDS):.1%}')
for threshold in (None, 0.7, 0.9):
    r = policy.evaluate(SqlDesk, SEEDS, threshold=threshold)
    label = f'defer below {threshold}' if threshold else 'never defers'
    print(f"model, {label:15} success {r['mean_reward']:.1%}, decisions handled {r['decisions_handled']:.0%}")

## 5. In your assistant loop

Before each LLM turn, ask the model; when it's confident, give your LLM only the tables it chose.

In [ ]:
desk = SqlDesk().reset(10_000_123)   # a conversation in progress: one catalog query so far
desk.query("SELECT name FROM sqlite_master WHERE type = 'table'")
print('user:', desk.question)

state = spec.state(desk.messages)
step = policy.decide(state, 'action', threshold=0.8)
print('next:', step.choice_id, f'(confidence {step.confidence:.2f})', 'DEFER' if step.defer else '')
if step.choice_id == 'run_sql' and not step.defer:
    tables = policy.decide(dict(state, next_call='run_sql'), 'tables.sql', threshold=0.8)
    print('tables:', tables.payload, f'(confidence {tables.confidence:.2f})', 'DEFER' if tables.defer else '')
    if not tables.defer and tables.payload != 'schema':
        print('schema for the LLM prompt:', {t: SCHEMA[t]['columns'] for t in tables.payload})

## What carries over

- **Any tool that takes SQL** works: put its argument name in `sql_params`. Other tools in the
  same logs become `action` candidates automatically.
- **Outcomes are the lever.** Filter on them for SFT. If your assistant is systematically wrong
  somewhere, imitation keeps the mistake: wrap a copy of your database in an `Environment` (see
  `SqlDesk`) and use `improve` ([cookbook 02](../02-shopdesk-traces-to-rl/cookbook.ipynb)).